# CPV301 D1 Pilot R5 packaged-state finalizer

Verification-only successor for the already packaged `epoch-0100` pilot. It does not train, resume training, rebuild a result archive, export the sealed test, or evaluate model quality. It authenticates the exact Drive artifacts and retry history through independently QA-passed R5 source.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import hashlib
import os
import stat
from pathlib import Path

TRAINING_COMMIT = "939f6bbce9f5c955b13ebcc84addd5083d242355"
R3_COMMIT = "e20a69541e3a6327739238fc71691109ce8f51d0"
R3_SHA256 = "8c9b23472091f6c677d635a54e4f4e8b896e1fcf204d627ba56eb208db00c49c"
R4_COMMIT = "2cc45b467456e9300df613839d94c5cb0bbdcbba"
R4_SHA256 = "ffcc88e2339b349775ca34e5e30531a102d0bcab1d45fa0dff9b1ea8bead25ad"
R5_COMMIT = "6acc47f08dba385b730ac86d591cb55248d104d2"
R5_SHA256 = "d4c7ab9b96fe92dd3eb6034fd1fb1144df7de35fbda88138f9b1d938f167d13b"
REPO_URL = "https://github.com/ScottPhantom/Traffic_SIgn_Detection.git"
REPO_DIR = Path("/content/Traffic_Sign_Detector")
STATE_DIR = Path(
    "/content/drive/MyDrive/CPV301/Traffic_Sign_Detector/"
    "M1_D1_Colab/runs/pilot_seed_20261018_r2"
)
R3_RUNNER = Path("/content/run_d1_colab_pilot_r3.py")
R4_RUNNER = Path("/content/run_d1_colab_pilot_r4.py")
R5_RUNNER = Path("/content/run_d1_colab_pilot_r5.py")

def verified_bytes(path, expected_sha256):
    fd = os.open(path, os.O_RDONLY | os.O_NOFOLLOW)
    try:
        info = os.fstat(fd)
        assert stat.S_ISREG(info.st_mode), "Runner is not regular"
        chunks = []
        while True:
            chunk = os.read(fd, 1024 * 1024)
            if not chunk:
                break
            chunks.append(chunk)
    finally:
        os.close(fd)
    payload = b"".join(chunks)
    assert hashlib.sha256(payload).hexdigest() == expected_sha256
    return payload


In [ ]:
import json
import subprocess
import urllib.request

latest = json.loads((STATE_DIR / "recovery/latest.json").read_text())
execution = json.loads((STATE_DIR / "colab_execution.json").read_text())
assert latest["generation"] == "epoch-0100", latest
assert execution["status"] == "SUCCEEDED", execution
assert execution["sealed_test_accessed"] is False

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)],
        check=True,
    )
subprocess.run(
    ["git", "checkout", "--detach", TRAINING_COMMIT],
    cwd=REPO_DIR,
    check=True,
)
observed = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert observed == TRAINING_COMMIT

def download_verified(path, commit, expected_sha256):
    url = (
        "https://raw.githubusercontent.com/ScottPhantom/"
        "Traffic_SIgn_Detection/"
        f"{commit}/detector/colab/{path.name}"
    )
    payload = urllib.request.urlopen(url, timeout=60).read()
    assert hashlib.sha256(payload).hexdigest() == expected_sha256
    if os.path.lexists(path):
        assert verified_bytes(path, expected_sha256) == payload
    else:
        flags = os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW
        fd = os.open(path, flags, 0o600)
        with os.fdopen(fd, "wb") as handle:
            handle.write(payload)
        assert verified_bytes(path, expected_sha256) == payload
    return payload

download_verified(R3_RUNNER, R3_COMMIT, R3_SHA256)
download_verified(R4_RUNNER, R4_COMMIT, R4_SHA256)
r5_bytes = download_verified(R5_RUNNER, R5_COMMIT, R5_SHA256)
print({
    "training_commit": observed,
    "r5_commit": R5_COMMIT,
    "r5_sha256": R5_SHA256,
    "latest_generation": latest["generation"],
    "sealed_test_accessed": False,
})


In [ ]:
import json
import subprocess
import sys

command = [
    sys.executable,
    "-",
    "--repo-root",
    str(REPO_DIR),
    "--state-dir",
    str(STATE_DIR),
]
completed = subprocess.run(command, input=r5_bytes, capture_output=True)
print(completed.stdout.decode("utf-8"))
if completed.stderr:
    print(completed.stderr.decode("utf-8"), file=sys.stderr)
completed.check_returncode()
result = json.loads(completed.stdout.decode("utf-8"))
assert result["status"] == "D1_PILOT_R5_PACKAGED_STATE_VERIFIED"
assert result["latest_generation"] == "epoch-0100"
assert result["runtime_evidence"]["attempt_count"] == 6
assert result["runtime_evidence"]["failed_resume_count"] == 4
assert result["runtime_evidence"]["distinct_runtime_digests"] is True
assert result["sealed_test_accessed"] is False
print("D1_PILOT_R5_COLAB_FINALIZER_PASS")
